# Evaluation and Optimization of IP DPU for Embedded AI/ML Inferencing
## Complete Analysis & Benchmark Report for Kria KV260

**Project Objective:** Build a representative audio AI/ML pipeline (GEMM-based keyword spotting) and benchmark the workload in **CPU-only**, **CPU + DPU (DPUCZDX8G)**, and **CPU + DPU + Custom HLS Kernel** modes on the AMD Kria KV260, backed by analytical modeling, software profiling, and hardware feasibility.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from pipeline.gemm_reference import build_mac_table, print_mac_table
from pipeline.dpu_performance_model import evaluate_analytical_model, KV260_DPU_B3136, KV260_CORTEX_A53
from pipeline.preprocessing import extract_log_mel, load_wav

## 1. GEMM-Based Workload Expression & MAC Counts
The full pipeline (Mel filterbank + DS-CNN Conv/DW/PW + FC) is expressed as pure matrix multiplications.

In [ ]:
print_mac_table('medium')

## 2. Analytical DPU Performance & Roofline Model
Evaluates layer-by-layer compute time, DDR bandwidth transfer time, and arithmetic intensity.

In [ ]:
estimates = evaluate_analytical_model('medium', hw_spec=KV260_DPU_B3136)
df_estimates = pd.DataFrame([{
    'Layer': e.layer_name,
    'MACs': f'{e.macs:,}',
    'AI (M/B)': f'{e.arithmetic_intensity:.2f}',
    'DPU Latency (us)': f'{e.dpu_latency_us:.1f}',
    'CPU Latency (us)': f'{e.cpu_latency_us:.1f}',
    'Speedup': f'{e.dpu_speedup:.2f}x',
    'Bound': e.bound_by,
} for e in estimates])
df_estimates

## 3. End-to-End Pipeline Latency Comparison (Config A vs B vs C)
Visualizing Amdahl's Law and stage transitions.

In [ ]:
from PIL import Image
img_breakdown = Image.open('../results/plots/latency_breakdown.png')
plt.figure(figsize=(10, 8))
plt.imshow(img_breakdown)
plt.axis('off')
plt.show()

## 4. Multi-Threaded Throughput Scaling
Evaluating inferences per second (FPS) scaling across 1, 2, and 4 threads.

In [ ]:
img_fps = Image.open('../results/plots/throughput_scaling.png')
plt.figure(figsize=(10, 6))
plt.imshow(img_fps)
plt.axis('off')
plt.show()

## 5. Summary & Partition Decision
Offloading DS-CNN neural inference to **DPUCZDX8G** yields a **~10.5x inference speedup**. Offloading the Mel filterbank GEMM to an **HLS PL stream kernel** eliminates the preprocessing bottleneck, delivering an overall **~9.2x end-to-end acceleration**.